## Marco del análisis

- **Qué estimamos:** la elasticidad precio-demanda propia de Pepsi 2L.
- **Target y grano:** ln(unidades) a nivel tienda·semana (panel, sin agregar).
- **Pregunta de pricing:** optimizar ingreso (el margen queda como posible extensión).
- **Confusores a controlar:** promoción, festivos, estacionalidad y tienda.
- **Naturaleza de los datos:** observacionales; la elasticidad es una asociación condicionada a los controles, válida dentro del rango de precios histórico. Amenaza principal: endogeneidad del precio.
- **Enfoque:** inferencia (calidad del coeficiente), no predicción. Partimos de la regresión ingenua y añadimos controles paso a paso.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


import statsmodels.formula.api as smf

### Carga de datos

In [2]:
pepsi = pd.read_parquet('../data/processed/pepsi_2l.parquet')
print(pepsi.shape)
print(pepsi.columns.tolist())

(31751, 15)
['STORE', 'UPC', 'WEEK', 'MOVE', 'QTY', 'PRICE', 'SALE', 'PROFIT', 'DESCRIP', 'SIZE', 'COM_CODE', 'start', 'end', 'special', 'UNIT_PRICE']


In [3]:
pepsi['UNIT_PRICE'] = pepsi['PRICE'] / pepsi['QTY'] # calculamos precio unitario
# aplicamos logaritmo a unidades vendidas y precio unitario para calcular elasticidad
pepsi['ln_q'] = np.log(pepsi['MOVE'])
pepsi['ln_p'] = np.log(pepsi['UNIT_PRICE'])

In [4]:
# Transformamos en variable dummy las promociones y festivos
pepsi['PROMO'] = pepsi['SALE'].notna().astype(int)
pepsi['FESTIVO'] = pepsi['special'].str.strip().notna().astype(int)

# Transformamos variable tiendas en categórica
pepsi['STORE'] = pepsi['STORE'].astype('category')


Para trabajar con las variables `SALE` y `special`, las transformamos en variables binarias, ya que no nos interesa en este momento que es cada cosa, simplemente aislar su efecto de la forma más sencilla posible. En `SALE` se registra el tipo de descuento aplicado al producto y en `special` el tipo de festividad que hay en esa semana.

Finalmente, como las tiendas se registran como número, las transformamos en variable categórica para que llegado el momento de aplicar la regresión, no interprete que una tienda tiene más peso que otra por tener diferente número.

Con el dataset estructurado, pasamos a la fase de regresión, en donde se van a generar varias regresiones añadiendo cada vez más complejidad con el efecto que provocan las promociones y días festivos. De esta forma se puede ir evaluando como cambia la elasticidad.

## Regresión

In [12]:
# inicializamos lista para guardar resultados
resultados = []

# Regresión 1 - Ingenua. Relación precio-cantidad sin ningún control.

m0 = smf.ols('ln_q ~ ln_p', data=pepsi).fit() # instanciamos y ajustamos modelo
resultados.append(('1. Ingenua', m0.params['ln_p']))
print(f"Elasticidad ingenua: {m0.params['ln_p']:.4f}")

Elasticidad ingenua: -4.0217


In [11]:
# Regresión con promoción incluida
m1 = smf.ols('ln_q ~ ln_p + PROMO', data=pepsi).fit()
resultados.append(('2. Promoción', m1.params['ln_p']))
print(f"Elasticidad (con promo): {m1.params['ln_p']:.4f}")
print(f"Efecto promoción: {m1.params['PROMO']:.4f} (p = {m1.pvalues['PROMO']:.3f})")

Elasticidad (con promo): -4.0496
Efecto promoción: -0.0176 (p = 0.077)


Los resultados de los coeficientes en esta segunda regresión son bastante contradictorios con lo que se espera, que es una bajada de la elasticidad, contrariamente de lo que pasa, que es un aumento de la misma. Este resultado junto a la no significatividad del efecto promoción en la regresión contradice los hallazgos del EDA, donde las semanas en promoción vendían mucho más que aquellas que registraban un precio regular. 

In [ ]:
# Cuánto se solapan precio y promoción?
pepsi.groupby("PROMO")["ln_p"].describe()[["mean", "min", "max"]]

,mean,min,max
PROMO,,,
0,0.407507,-0.235722,0.636577
1,0.155039,-0.385662,0.636577


Como ya hemos codificado el efecto de promoción en binario, vamos a examinar esta variable más de cerca, ya que la variable original en la documentación de los propios datos nos indica que puede quedar algún error en las promociones. Si calculamos el precio medio, se puede observar como aquellos precios con promocion (`1`) tienen un precio mucho más bajo que los precios regulares. Observando el rango de precio, que ambas categorías coincidan exactamente en el valor máximo, nos indica lo que ya se sospechaba e indicaba la documentación, posibles errores en el registro de promociones.

In [14]:
pepsi.groupby("PROMO")["MOVE"].mean()

PROMO
0    161.755329
1    508.268573
Name: MOVE, dtype: float64

In [16]:
# Semanas marcadas como promo pero con precio alto: ¿tienen sentido?
pepsi[pepsi["PROMO"] == 1].sort_values("ln_p", ascending=False)[
    ["STORE", "start", "UNIT_PRICE", "MOVE", "SALE", "PROMO"]
].head(10)

,STORE,start,UNIT_PRICE,MOVE,SALE,PROMO
210219,14,1991-07-11,1.89,44,S,1
224222,130,1991-07-11,1.89,157,S,1
213094,53,1991-07-11,1.89,23,S,1
218704,95,1991-07-11,1.89,107,S,1
214435,68,1991-07-11,1.89,87,S,1
214820,71,1991-07-11,1.89,62,S,1
215015,72,1991-07-11,1.89,40,S,1
213876,62,1991-07-11,1.89,30,S,1
218323,93,1991-07-11,1.89,51,S,1
225045,137,1991-07-11,1.89,48,S,1


Inspeccionando las semanas marcadas como promción podemos ver la causa, hay registros marcados como promción pero que cuentan con el precio más elevado y el número de ventas es bajo.

Para tratar de corregir esta situación vamos a reconstruir la señal de promoción a partir del propio precio. Para reconstruir el precio lo que se hace es definir un **precio de referencia** por tienda, el precio regular del producto en cada tienda, y se mide la **profundiad de descuento** como la caída del precio observado respecto a la referencia. 

Para construir el precio de referencia vamos a emplear un cuantil alto (percentil 90), es que donde se encuentran los precios altos, para una ventana móvil de 13 semanas, haciendolo coincidir con el trimestre, calculado por tienda. La lógica de esta decisión es que como las promociones bajan el precio y son frecuentes en la serie, el precio regular siempre vive en la parte alta de la distribución local. Por tanto un cuantil alto captura mejor que la mediana, que se vería contaminada con la cantidad de semanas que tiene el precio descontado. La ventana móvil permite que la referencia de precio se adapte al cambio que se vaya ocasionando a lo largo de la serie, de esta forma evitamos establecer un precio fijo para los 6 años.